In [1]:
# ==========================================================
# BURUSHASKI DICTIONARY PARSER
#
# PROJECT CONFIGURATION
#
# Run this cell ONCE.
# Every later pass imports these variables.
# ==========================================================

from pathlib import Path
import os
import json
import re
import unicodedata
import logging

# ----------------------------------------------------------
# Project folders
# ----------------------------------------------------------

PROJECT_ROOT = Path("/kaggle/working")

OUTPUT_DIR = PROJECT_ROOT / "output"

CACHE_DIR = PROJECT_ROOT / "cache"

LOG_DIR = PROJECT_ROOT / "logs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# PDF
# ----------------------------------------------------------

PDF_PATH = Path(
    "/kaggle/input/datasets/ibrahim112003/burushovacab/Burushaski A Reference Grammar of Eastern (Yoshioka).pdf"
)

START_PAGE = 503
END_PAGE = 617

# ----------------------------------------------------------
# Output files
# ----------------------------------------------------------

RAW_LINES_FILE = OUTPUT_DIR / "01_raw_lines.txt"

LOGICAL_LINES_FILE = OUTPUT_DIR / "02_logical_lines.txt"

TOKENS_FILE = OUTPUT_DIR / "03_tokens.json"

TOKENS_PRETTY_FILE = OUTPUT_DIR / "03_tokens_pretty.txt"

ENTRIES_FILE = OUTPUT_DIR / "04_entries.json"

CLASSIFIED_FILE = OUTPUT_DIR / "05_classified.json"

PARSED_FILE = OUTPUT_DIR / "06_parsed.json"

FINAL_JSON = OUTPUT_DIR / "burushaski_dictionary.json"

SEARCH_JSON = OUTPUT_DIR / "burushaski_search.json"

# ----------------------------------------------------------
# Logging
# ----------------------------------------------------------

logging.basicConfig(

    filename=LOG_DIR / "pipeline.log",

    level=logging.INFO,

    format="%(asctime)s %(levelname)s %(message)s"
)

logging.info("Pipeline started")

print("="*60)
print("PROJECT INITIALIZED")
print("="*60)

print()

print("Project root :", PROJECT_ROOT)

print("Output folder:", OUTPUT_DIR)

print()

print("Files will be written to:")

for f in [

    RAW_LINES_FILE,

    LOGICAL_LINES_FILE,

    TOKENS_FILE,

    ENTRIES_FILE,

    CLASSIFIED_FILE,

    PARSED_FILE,

    FINAL_JSON
]:

    print(" -", f)

print()

print("READY")

PROJECT INITIALIZED

Project root : /kaggle/working
Output folder: /kaggle/working/output

Files will be written to:
 - /kaggle/working/output/01_raw_lines.txt
 - /kaggle/working/output/02_logical_lines.txt
 - /kaggle/working/output/03_tokens.json
 - /kaggle/working/output/04_entries.json
 - /kaggle/working/output/05_classified.json
 - /kaggle/working/output/06_parsed.json
 - /kaggle/working/output/burushaski_dictionary.json

READY


In [3]:
# ==========================================================
# CELL 2
# PASS 1
#
# BURUSHASKI DICTIONARY PIPELINE
#
# PURPOSE
# ----------------------------------------------------------
# 1. Read PDF
# 2. Remove thesis artifacts
# 3. Normalize Unicode
# 4. Preserve every extracted line
# 5. Number every line
#
# OUTPUT
#
# output/
#     01_raw_lines.txt
#     01_statistics.json
#
# ==========================================================

!pip install pymupdf -q

from pathlib import Path
import fitz
import unicodedata
import re
import json
import logging
from tqdm import tqdm

# ==========================================================
# CONFIGURATION
# ==========================================================

PDF_PATH = Path(
    "/kaggle/input/datasets/ibrahim112003/burushovacab/"
    "Burushaski A Reference Grammar of Eastern (Yoshioka).pdf"
)

START_PAGE = 503
END_PAGE = 617

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_OUTPUT = OUTPUT_DIR / "01_raw_lines.txt"
STAT_OUTPUT = OUTPUT_DIR / "01_statistics.json"

# ==========================================================
# LOGGER
# ==========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s"
)

log = logging.getLogger("PASS1")

# ==========================================================
# REGEX
# ==========================================================

ROMAN_PAGE = re.compile(r'^[CLXVI]+$')

THESIS = re.compile(
    r'Doctoral Thesis',
    re.IGNORECASE
)

JAPANESE = re.compile(
    r'[\u3040-\u30ff\u4e00-\u9fff]'
)

MULTISPACE = re.compile(r'[ \t]+')


# ==========================================================
# CLEAN ONE LINE
# ==========================================================

def clean_line(line: str) -> str:

    line = unicodedata.normalize("NFC", line)

    line = MULTISPACE.sub(" ", line)

    return line.strip()


# ==========================================================
# ARTIFACT DETECTOR
# ==========================================================

def is_artifact(line: str) -> bool:

    if not line:
        return True

    if THESIS.search(line):
        return True

    if JAPANESE.search(line):
        return True

    if ROMAN_PAGE.fullmatch(line):
        return True

    return False


# ==========================================================
# EXTRACT LINES
# ==========================================================

def extract_lines():

    doc = fitz.open(PDF_PATH)

    lines = []

    removed = 0

    log.info("Reading PDF...\n")

    for page_no in tqdm(range(START_PAGE, END_PAGE + 1)):

        if page_no >= len(doc):
            break

        page = doc.load_page(page_no)

        text = page.get_text("text")

        for raw in text.splitlines():

            line = clean_line(raw)

            if is_artifact(line):
                removed += 1
                continue

            lines.append(line)

    doc.close()

    return lines, removed


# ==========================================================
# WRITE OUTPUT
# ==========================================================

def save_lines(lines):

    with open(RAW_OUTPUT, "w", encoding="utf8") as f:

        for idx, line in enumerate(lines, start=1):

            f.write(f"{idx:06d}\t{line}\n")


# ==========================================================
# SAVE STATS
# ==========================================================

def save_stats(lines, removed):

    stats = {

        "pages_processed": END_PAGE - START_PAGE + 1,

        "lines_written": len(lines),

        "artifacts_removed": removed,

        "output_file": str(RAW_OUTPUT)

    }

    with open(STAT_OUTPUT, "w", encoding="utf8") as f:

        json.dump(stats, f, indent=4)

    return stats


# ==========================================================
# MAIN
# ==========================================================

raw_lines, removed = extract_lines()

save_lines(raw_lines)

stats = save_stats(raw_lines, removed)

print("\n" + "=" * 60)
print("PASS 1 COMPLETE")
print("=" * 60)

print(f"Pages processed : {stats['pages_processed']}")
print(f"Lines written   : {stats['lines_written']}")
print(f"Artifacts       : {stats['artifacts_removed']}")

print("\nGenerated files")
print(RAW_OUTPUT)
print(STAT_OUTPUT)

100%|██████████| 115/115 [00:00<00:00, 222.78it/s]


PASS 1 COMPLETE
Pages processed : 115
Lines written   : 6278
Artifacts       : 487

Generated files
output/01_raw_lines.txt
output/01_statistics.json


In [4]:
# ==========================================================
# PASS 2
#
# LEXICAL LINE NORMALIZATION
#
# PURPOSE
# ----------------------------------------------------------
# 1. Read raw extracted lines
# 2. Normalize whitespace
# 3. Normalize unicode
# 4. Normalize dashes
# 5. Preserve one logical dictionary line per record
#
# INPUT
#
# output/01_raw_lines.txt
#
# OUTPUT
#
# output/02_normalized_lines.txt
# output/02_statistics.json
#
# ==========================================================

from pathlib import Path
import unicodedata
import re
import json

# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------

OUTPUT_DIR = Path("output")

INPUT_FILE = OUTPUT_DIR / "01_raw_lines.txt"

OUTPUT_FILE = OUTPUT_DIR / "02_normalized_lines.txt"

STAT_FILE = OUTPUT_DIR / "02_statistics.json"

# ----------------------------------------------------------
# REGEX
# ----------------------------------------------------------

MULTISPACE = re.compile(r"[ \t]+")

MULTIDASH = re.compile(r"[‐-–—]+")

# ----------------------------------------------------------
# NORMALIZE
# ----------------------------------------------------------

def normalize_line(line):

    line = unicodedata.normalize("NFC", line)

    line = MULTISPACE.sub(" ", line)

    line = MULTIDASH.sub("-", line)

    return line.strip()

# ----------------------------------------------------------
# READ
# ----------------------------------------------------------

normalized = []

input_count = 0

with open(INPUT_FILE, "r", encoding="utf8") as f:

    for row in f:

        input_count += 1

        row = row.rstrip()

        if not row:
            continue

        number, text = row.split("\t", 1)

        text = normalize_line(text)

        if text == "":
            continue

        normalized.append((number, text))

# ----------------------------------------------------------
# WRITE
# ----------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf8") as f:

    for number, text in normalized:

        f.write(f"{number}\t{text}\n")

# ----------------------------------------------------------
# STATS
# ----------------------------------------------------------

stats = {

    "input_lines": input_count,

    "normalized_lines": len(normalized),

    "output_file": str(OUTPUT_FILE)

}

with open(STAT_FILE, "w", encoding="utf8") as f:

    json.dump(stats, f, indent=4)

print("=" * 60)
print("PASS 2 COMPLETE")
print("=" * 60)

print("Input lines      :", input_count)
print("Normalized lines :", len(normalized))

print("\nGenerated")

print(OUTPUT_FILE)

print(STAT_FILE)

PASS 2 COMPLETE
Input lines      : 6278
Normalized lines : 6278

Generated
output/02_normalized_lines.txt
output/02_statistics.json


In [6]:
# ============================================================
# PASS 3A
# LINE TYPE DETECTOR
# ============================================================

import re
import json
from pathlib import Path
from collections import Counter

INPUT = OUTPUT_DIR / "02_normalized_lines.txt"

OUTPUT_JSON = OUTPUT_DIR / "03_line_types.json"

OUTPUT_TXT = OUTPUT_DIR / "03_line_types.txt"

# ------------------------------------------------------------
# POS TAGS
# ------------------------------------------------------------

POS = {
    "X","Y","Z","H",
    "HM","HF","HS","HX",
    "ADJ","ADV",
    "PRN","NUM",
    "CONJ","INTERJ",
    "ONO",
    "TR","INTR"
}

# ------------------------------------------------------------
# REGEX
# ------------------------------------------------------------

HEADWORD_RE = re.compile(
    r"^[^\s].+?\b(" + "|".join(POS) + r")\b"
)

VERB_RE = re.compile(
    r"\b(INTR|TR)\b"
)

SEE_RE = re.compile(
    r"\bsee\b",
    re.I
)

REFERENCE_RE = re.compile(
    r"\|\|"
)

BORROW_RE = re.compile(
    r"^¶"
)

PL_RE = re.compile(
    r"^(PL|SG|DOUBLE|NEG|IPFV|PFV|CP|HZ|NG)\b"
)

EXAMPLE_RE = re.compile(
    r"^[a-zA-Z@].+?-"
)

# ------------------------------------------------------------
# CLASSIFIER
# ------------------------------------------------------------

def classify(line):

    line=line.strip()

    if not line:
        return "EMPTY"

    if SEE_RE.search(line):
        return "CROSS_REFERENCE"

    if BORROW_RE.match(line):
        return "BORROWING"

    if REFERENCE_RE.search(line):
        return "REFERENCE"

    if VERB_RE.search(line):
        return "VERB"

    if HEADWORD_RE.search(line):
        return "HEADWORD"

    if PL_RE.search(line):
        return "GRAMMAR"

    if EXAMPLE_RE.match(line):
        return "EXAMPLE"

    return "UNKNOWN"

# ------------------------------------------------------------
# READ
# ------------------------------------------------------------

lines=[]

with open(INPUT,encoding="utf8") as f:

    for row in f:

        row=row.rstrip()

        if not row:
            continue

        number,text=row.split("\t",1)

        lines.append((int(number),text))

# ------------------------------------------------------------
# PROCESS
# ------------------------------------------------------------

results=[]

counter=Counter()

for no,text in lines:

    t=classify(text)

    counter[t]+=1

    results.append({

        "line":no,

        "type":t,

        "text":text

    })

# ------------------------------------------------------------
# WRITE JSON
# ------------------------------------------------------------

with open(OUTPUT_JSON,"w",encoding="utf8") as f:

    json.dump(results,f,indent=2,ensure_ascii=False)

# ------------------------------------------------------------
# WRITE HUMAN FILE
# ------------------------------------------------------------

with open(OUTPUT_TXT,"w",encoding="utf8") as f:

    for r in results:

        f.write(
            f"[{r['type']:<18}] {r['text']}\n"
        )

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("="*60)
print("PASS 3A COMPLETE")
print("="*60)

for k,v in counter.items():

    print(f"{k:<20} {v}")

print()

print(OUTPUT_JSON)

print(OUTPUT_TXT)

PASS 3A COMPLETE
REFERENCE            2478
UNKNOWN              2709
EXAMPLE              731
HEADWORD             53
BORROWING            16
CROSS_REFERENCE      223
VERB                 61
GRAMMAR              7

output/03_line_types.json
output/03_line_types.txt


In [7]:
# ============================================================
# PASS 3.1
# ENTRY BOUNDARY DETECTOR
#
# PURPOSE
# -------
# Reconstruct complete dictionary entries.
#
# INPUT
# -----
# output/02_normalized_lines.txt
#
# OUTPUT
# ------
# output/03_1_entries.json
# output/03_1_entries.txt
# output/03_1_statistics.json
# ============================================================

import json
import re
from pathlib import Path
from statistics import mean

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------

INPUT_FILE = OUTPUT_DIR / "02_normalized_lines.txt"

ENTRY_JSON = OUTPUT_DIR / "03_1_entries.json"

ENTRY_TEXT = OUTPUT_DIR / "03_1_entries.txt"

STAT_FILE = OUTPUT_DIR / "03_1_statistics.json"

# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

POS_TAGS = {

    "X","Y","Z","H",

    "HM","HF","HS","HX",

    "ADJ","ADV",

    "NUM","PRN",

    "CONJ","INTERJ",

    "ONO",

    "TR","INTR"

}

GRAMMAR_PREFIXES = (

    "PL",

    "SG",

    "DOUBLE",

    "NEG",

    "IPFV",

    "PFV",

    "CP",

    "HZ",

    "NG"

)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def contains_pos(line):

    words = line.split()

    for w in words:

        if w in POS_TAGS:

            return True

    return False


def begins_with_grammar(line):

    first = line.split()[0]

    return first in GRAMMAR_PREFIXES


def is_reference(line):

    return line.startswith("||")


def is_borrowing(line):

    return line.startswith("¶")


def is_cross_reference(line):

    return " see " in f" {line} "


# ------------------------------------------------------------
# ENTRY SCORING
# ------------------------------------------------------------

def entry_score(line):

    score = 0

    line = line.strip()

    if not line:

        return -100

    # ----------------------------------------
    # definitely NOT a new entry
    # ----------------------------------------

    if is_reference(line):

        score -= 10

    if is_borrowing(line):

        score -= 8

    if begins_with_grammar(line):

        score -= 6

    # ----------------------------------------
    # strong indicators
    # ----------------------------------------

    if contains_pos(line):

        score += 4

    if " INTR " in f" {line} ":

        score += 5

    if " TR " in f" {line} ":

        score += 5

    if is_cross_reference(line):

        score += 3

    # lexical-looking first token

    first = line.split()[0]

    if len(first) > 1:

        score += 2

    # long descriptive sentence?
    # probably continuation

    if len(line.split()) > 18:

        score -= 2

    return score


def is_new_entry(line):

    return entry_score(line) >= 4


# ------------------------------------------------------------
# READ
# ------------------------------------------------------------

lines = []

with open(INPUT_FILE, encoding="utf8") as f:

    for row in f:

        row = row.rstrip()

        if not row:

            continue

        number, text = row.split("\t", 1)

        lines.append((int(number), text))

# ------------------------------------------------------------
# BUILD ENTRIES
# ------------------------------------------------------------

entries = []

current = None

for line_no, text in lines:

    if current is None:

        current = {

            "entry_id": 1,

            "start_line": line_no,

            "end_line": line_no,

            "lines": [text]

        }

        continue

    if is_new_entry(text):

        entries.append(current)

        current = {

            "entry_id": len(entries)+1,

            "start_line": line_no,

            "end_line": line_no,

            "lines": [text]

        }

    else:

        current["end_line"] = line_no

        current["lines"].append(text)

# last entry

if current:

    entries.append(current)

# ------------------------------------------------------------
# WRITE JSON
# ------------------------------------------------------------

with open(ENTRY_JSON,"w",encoding="utf8") as f:

    json.dump(
        entries,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# WRITE HUMAN FILE
# ------------------------------------------------------------

with open(ENTRY_TEXT,"w",encoding="utf8") as f:

    for e in entries:

        f.write("="*70+"\n")

        f.write(f"ENTRY {e['entry_id']}\n")

        f.write(
            f"LINES {e['start_line']} - {e['end_line']}\n"
        )

        f.write("="*70+"\n\n")

        for line in e["lines"]:

            f.write(line+"\n")

        f.write("\n\n")

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

sizes = [len(e["lines"]) for e in entries]

stats = {

    "entries": len(entries),

    "largest_entry": max(sizes),

    "smallest_entry": min(sizes),

    "average_lines": round(mean(sizes),2)

}

with open(STAT_FILE,"w",encoding="utf8") as f:

    json.dump(stats,f,indent=4)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("="*60)

print("PASS 3.1 COMPLETE")

print("="*60)

print(f"Input lines      : {len(lines)}")

print(f"Entries detected : {len(entries)}")

print(f"Largest entry    : {stats['largest_entry']}")

print(f"Smallest entry   : {stats['smallest_entry']}")

print(f"Average lines    : {stats['average_lines']}")

print()

print("Generated")

print(ENTRY_JSON)

print(ENTRY_TEXT)

print(STAT_FILE)

PASS 3.1 COMPLETE
Input lines      : 6278
Entries detected : 2616
Largest entry    : 34
Smallest entry   : 1
Average lines    : 2.4

Generated
output/03_1_entries.json
output/03_1_entries.txt
output/03_1_statistics.json


In [8]:
# ============================================================
# PASS 3.2
#
# LEXICAL FAMILY BUILDER
#
# INPUT
# -----
# output/03_1_entries.json
#
# OUTPUT
# ------
# output/03_2_families.json
# output/03_2_families.txt
# output/03_2_statistics.json
#
# PURPOSE
# -------
# Split every dictionary entry into logical blocks.
#
# No parsing yet.
#
# ============================================================

import json
import re
from pathlib import Path
from statistics import mean

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

OUTPUT = Path("output")

INPUT = OUTPUT / "03_1_entries.json"

JSON_OUT = OUTPUT / "03_2_families.json"

TXT_OUT = OUTPUT / "03_2_families.txt"

STAT_OUT = OUTPUT / "03_2_statistics.json"

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

with open(INPUT,"r",encoding="utf8") as f:

    entries = json.load(f)

# ------------------------------------------------------------
# REGEX
# ------------------------------------------------------------

POS = re.compile(
    r'\b('
    r'ADJ|ADV|CONJ|INTERJ|NUM|PRN|POST|PREP|'
    r'HM|HF|H|X|Y|Z|TR|INTR'
    r')\b'
)

REFERENCE = re.compile(r'\bsee\b',re.I)

BORROWING = re.compile(r'¶')

GRAMMAR = re.compile(
    r'\b('
    r'IPFV|PFV|NEG|CP|PL|SG|ERG|GEN|DAT|ABL|ADE|PROX|DIST|OBJ'
    r')\b'
)

# starts with derivational prefixes

DERIVED = re.compile(

    r'^(@-|d-|- )'

)

# phrase usually starts with two lexical words

PHRASE = re.compile(

    r'^[^\s]+\s+[^\s]+'

)

# ------------------------------------------------------------
# BLOCK CLASSIFIER
# ------------------------------------------------------------

def classify(line):

    line=line.strip()

    if not line:

        return "EMPTY"

    if REFERENCE.search(line):

        return "CROSS_REFERENCE"

    if BORROWING.search(line):

        return "BORROWING"

    if POS.search(line):

        return "MAIN"

    if DERIVED.match(line):

        return "DERIVED"

    if GRAMMAR.search(line):

        return "CONTINUATION"

    if PHRASE.match(line):

        return "PHRASE"

    return "CONTINUATION"

# ------------------------------------------------------------
# FAMILY BUILDER
# ------------------------------------------------------------

families=[]

for entry in entries:

    blocks=[]

    current=None

    for line in entry["lines"]:

        kind=classify(line)

        # ------------------------------------------
        # start new lexical block
        # ------------------------------------------

        if kind in {

            "MAIN",
            "DERIVED",
            "PHRASE",
            "CROSS_REFERENCE"

        }:

            if current:

                blocks.append(current)

            current={

                "type":kind,

                "lines":[line]

            }

        else:

            if current is None:

                current={

                    "type":"MAIN",

                    "lines":[]

                }

            current["lines"].append(line)

    if current:

        blocks.append(current)

    families.append({

        "family_id":entry["entry_id"],

        "blocks":blocks

    })

# ------------------------------------------------------------
# WRITE JSON
# ------------------------------------------------------------

with open(JSON_OUT,"w",encoding="utf8") as f:

    json.dump(

        families,

        f,

        ensure_ascii=False,

        indent=2

    )

# ------------------------------------------------------------
# WRITE TXT
# ------------------------------------------------------------

with open(TXT_OUT,"w",encoding="utf8") as f:

    for fam in families:

        f.write("="*70+"\n")

        f.write(f"FAMILY {fam['family_id']}\n\n")

        for i,block in enumerate(fam["blocks"],1):

            f.write(f"[BLOCK {i}] {block['type']}\n")

            for line in block["lines"]:

                f.write("    "+line+"\n")

            f.write("\n")

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

block_counts=[len(f["blocks"]) for f in families]

stats={

    "families":len(families),

    "total_blocks":sum(block_counts),

    "average_blocks_per_family":round(mean(block_counts),2),

    "largest_family":max(block_counts),

    "smallest_family":min(block_counts)

}

with open(STAT_OUT,"w",encoding="utf8") as f:

    json.dump(stats,f,indent=4)

print("="*60)
print("PASS 3.2 COMPLETE")
print("="*60)

print(f"Families        : {stats['families']}")
print(f"Blocks          : {stats['total_blocks']}")
print(f"Average Blocks  : {stats['average_blocks_per_family']}")
print(f"Largest Family  : {stats['largest_family']}")
print(f"Smallest Family : {stats['smallest_family']}")

print("\nGenerated")
print(JSON_OUT)
print(TXT_OUT)
print(STAT_OUT)

PASS 3.2 COMPLETE
Families        : 2616
Blocks          : 4669
Average Blocks  : 1.78
Largest Family  : 28
Smallest Family : 1

Generated
output/03_2_families.json
output/03_2_families.txt
output/03_2_statistics.json


In [9]:
# ============================================================
# PASS 4
# ENTRY PACKAGING
#
# INPUT
# -----
# output/03_1_entries.json
#
# OUTPUT
# ------
# output/04_entry_objects.json
# output/04_entry_objects_pretty.txt
# output/04_statistics.json
# ============================================================

import json
from pathlib import Path
from statistics import mean

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

OUTPUT_DIR = Path("output")

INPUT_FILE = OUTPUT_DIR / "03_1_entries.json"

OUTPUT_JSON = OUTPUT_DIR / "04_entry_objects.json"

OUTPUT_TEXT = OUTPUT_DIR / "04_entry_objects_pretty.txt"

OUTPUT_STATS = OUTPUT_DIR / "04_statistics.json"

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf8") as f:
    entries = json.load(f)

# ------------------------------------------------------------
# BUILD OBJECTS
# ------------------------------------------------------------

objects = []

for idx, entry in enumerate(entries, start=1):

    lines = [
        x.strip()
        for x in entry
        if x.strip()
    ]

    if len(lines) == 0:
        continue

    obj = {

        "id": idx,

        "first_line": lines[0],

        "raw_lines": lines,

        "line_count": len(lines),

        # filled later
        "entry_type": None,

        "headword": None,

        "part_of_speech": None,

        "parser": None,

        "parsed": False,

        "fields": {},

        "errors": []

    }

    objects.append(obj)

# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

with open(OUTPUT_JSON, "w", encoding="utf8") as f:

    json.dump(
        objects,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# HUMAN READABLE
# ------------------------------------------------------------

with open(OUTPUT_TEXT, "w", encoding="utf8") as f:

    for obj in objects:

        f.write("=" * 80 + "\n")

        f.write(f"ENTRY {obj['id']}\n")

        f.write(f"LINES : {obj['line_count']}\n")

        f.write(f"FIRST : {obj['first_line']}\n")

        f.write("\nRAW\n")

        for line in obj["raw_lines"]:
            f.write("    " + line + "\n")

        f.write("\n")

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

stats = {

    "entries": len(objects),

    "average_lines": round(

        mean(
            o["line_count"]
            for o in objects
        ),
        2
    ),

    "largest_entry": max(

        o["line_count"]
        for o in objects
    ),

    "smallest_entry": min(

        o["line_count"]
        for o in objects
    )

}

with open(OUTPUT_STATS, "w", encoding="utf8") as f:

    json.dump(
        stats,
        f,
        indent=4
    )

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 60)

print("PASS 4 COMPLETE")

print("=" * 60)

print("Entries :", stats["entries"])

print("Average :", stats["average_lines"])

print("Largest :", stats["largest_entry"])

print("Smallest:", stats["smallest_entry"])

print()

print("Generated")

print(OUTPUT_JSON)

print(OUTPUT_TEXT)

print(OUTPUT_STATS)

PASS 4 COMPLETE
Entries : 2616
Average : 4
Largest : 4
Smallest: 4

Generated
output/04_entry_objects.json
output/04_entry_objects_pretty.txt
output/04_statistics.json


In [10]:
import json

with open("output/03_1_entries.json","r",encoding="utf8") as f:
    data=json.load(f)

print(type(data))
print(len(data))

print("\nFIRST OBJECT\n")
print(data[0])

<class 'list'>
2616

FIRST OBJECT

{'entry_id': 1, 'start_line': 1, 'end_line': 34, 'lines': ['(||)', ': Leading symbol for the information of other studies', '(AA.#00)', ': Basic word number according to Research Institute for Languages and', 'Cultures of Asia and Africa (ed.) (1967)', '(B.00)', ': Page number in Berger (1998c). Additionally, I give some information', 'with round brackets after citing the page number, when', '(i) the form of stem is not identical to Berger’s entry, then the relevant', 'form by Berger is given.', 'e.g. aasmáan ... B.22 (asmáan);', '(ii) the form is the same as Berger’s second (or lesser) entry, then I give his', 'first entry with the same entry as mine (i.e. the latter item is identical to', 'mine).', 'e.g. dšá ... B.25 ( ćá , d á );', '(iii) Berger attaches a superscript number to his entry (to make distinction', 'among synonymous words), I always specify it.', 'e.g. adáp ... B.12 (2adáp).', 'The absence of this content indicates the fact that there i

In [11]:
import json
from pprint import pprint

with open("output/03_1_entries.json","r",encoding="utf8") as f:
    data=json.load(f)

pprint(data[:3], width=120)

[{'end_line': 34,
  'entry_id': 1,
  'lines': ['(||)',
            ': Leading symbol for the information of other studies',
            '(AA.#00)',
            ': Basic word number according to Research Institute for Languages and',
            'Cultures of Asia and Africa (ed.) (1967)',
            '(B.00)',
            ': Page number in Berger (1998c). Additionally, I give some information',
            'with round brackets after citing the page number, when',
            '(i) the form of stem is not identical to Berger’s entry, then the relevant',
            'form by Berger is given.',
            'e.g. aasmáan ... B.22 (asmáan);',
            '(ii) the form is the same as Berger’s second (or lesser) entry, then I give his',
            'first entry with the same entry as mine (i.e. the latter item is identical to',
            'mine).',
            'e.g. dšá ... B.25 ( ćá , d á );',
            '(iii) Berger attaches a superscript number to his entry (to make distinction',
       

In [12]:
# ==========================================================
# PASS 5
# RULE-BASED PARSER ENGINE
# ==========================================================

import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional

INPUT_FILE = "output/03_1_entries.json"

OUTPUT_FILE = "output/05_stage0_engine.json"

# ==========================================================
# ENTRY OBJECT
# ==========================================================

@dataclass
class Entry:

    id: int

    raw_lines: List[str]

    # Filled gradually

    repaired_lines: List[str] = field(default_factory=list)

    headword: Optional[str] = None

    pos: List[str] = field(default_factory=list)

    entry_type: Optional[str] = None

    meanings: List[str] = field(default_factory=list)

    references: List[str] = field(default_factory=list)

    borrowings: List[str] = field(default_factory=list)

    synonyms: List[str] = field(default_factory=list)

    cross_refs: List[str] = field(default_factory=list)

    metadata: Dict = field(default_factory=dict)

# ==========================================================
# DETECTOR BASE CLASS
# ==========================================================

class Detector:

    def process(self, entry: Entry):

        raise NotImplementedError()

# ==========================================================
# REPAIR DETECTOR (placeholder)
# ==========================================================

class RepairDetector(Detector):

    def process(self, entry):

        entry.repaired_lines = list(entry.raw_lines)

# ==========================================================
# HEADWORD DETECTOR (placeholder)
# ==========================================================

class HeadwordDetector(Detector):

    def process(self, entry):

        pass

# ==========================================================
# POS DETECTOR (placeholder)
# ==========================================================

class POSTDetector(Detector):

    def process(self, entry):

        pass

# ==========================================================
# MEANING DETECTOR (placeholder)
# ==========================================================

class MeaningDetector(Detector):

    def process(self, entry):

        pass

# ==========================================================
# METADATA DETECTOR (placeholder)
# ==========================================================

class MetadataDetector(Detector):

    def process(self, entry):

        pass

# ==========================================================
# PARSER ENGINE
# ==========================================================

pipeline = [

    RepairDetector(),

    HeadwordDetector(),

    POSTDetector(),

    MeaningDetector(),

    MetadataDetector()

]

# ==========================================================
# LOAD ENTRIES
# ==========================================================

with open(INPUT_FILE,"r",encoding="utf8") as f:

    raw_entries = json.load(f)

entries = []

for e in raw_entries:

    entry = Entry(

        id=e["entry_id"],

        raw_lines=e["lines"]

    )

    for detector in pipeline:

        detector.process(entry)

    entries.append(entry)

# ==========================================================
# SAVE
# ==========================================================

output=[]

for e in entries:

    output.append({

        "id":e.id,

        "headword":e.headword,

        "pos":e.pos,

        "entry_type":e.entry_type,

        "meanings":e.meanings,

        "references":e.references,

        "borrowings":e.borrowings,

        "synonyms":e.synonyms,

        "cross_refs":e.cross_refs,

        "raw_lines":e.raw_lines,

        "repaired_lines":e.repaired_lines

    })

with open(OUTPUT_FILE,"w",encoding="utf8") as f:

    json.dump(output,f,ensure_ascii=False,indent=4)

print("="*60)
print("PASS 5 STAGE 0 COMPLETE")
print("="*60)
print("Entries :",len(output))
print("Generated:",OUTPUT_FILE)

PASS 5 STAGE 0 COMPLETE
Entries : 2616
Generated: output/05_stage0_engine.json


In [17]:
# ============================================================
# PASS 5.1
#
# Detector Engine
#
# Every detector receives an Entry object
# and modifies it.
#
# Nothing is parsed permanently here.
#
# ============================================================

import json
from pathlib import Path
from dataclasses import dataclass, field

INPUT = Path("/kaggle/working/output/05_stage0_engine.json")

OUTPUT = Path("output/05_1_detector_objects.json")

# ------------------------------------------------------------
# Entry object
# ------------------------------------------------------------

@dataclass
class Entry:

    id:int

    raw_lines:list

    repaired_lines:list = field(default_factory=list)

    headword=None

    pos=None

    grammar=dict()

    meanings=list()

    metadata=dict()

    confidence=0.0

    parser_state="RAW"

    errors=list()

# ------------------------------------------------------------
# Base detector
# ------------------------------------------------------------

class Detector:

    def process(self,entry):

        return entry

# ------------------------------------------------------------
# Pipeline
# ------------------------------------------------------------

class DetectorPipeline:

    def __init__(self):

        self.detectors=[]

    def add(self,detector):

        self.detectors.append(detector)

    def process(self,entry):

        for detector in self.detectors:

            detector.process(entry)

        return entry

In [19]:
# ------------------------------------------------------------
# Repair Detector
# ------------------------------------------------------------

class RepairDetector(Detector):

    def process(self,entry):

        repaired=[]

        for line in entry.raw_lines:

            repaired.append(line.strip())

        entry.repaired_lines=repaired

        entry.parser_state="REPAIRED"

        return entry

In [20]:
pipeline=DetectorPipeline()

pipeline.add(RepairDetector())

In [21]:
with open(INPUT,"r",encoding="utf8") as f:

    raw=json.load(f)

entries=[]

for e in raw:

    obj=Entry(

        id=e["id"],

        raw_lines=e["raw_lines"]

    )

    obj=pipeline.process(obj)

    entries.append(obj)

In [22]:
def serialize(entry):

    return {

        "id":entry.id,

        "raw_lines":entry.raw_lines,

        "repaired_lines":entry.repaired_lines,

        "headword":entry.headword,

        "pos":entry.pos,

        "grammar":entry.grammar,

        "meanings":entry.meanings,

        "metadata":entry.metadata,

        "confidence":entry.confidence,

        "parser_state":entry.parser_state,

        "errors":entry.errors

    }

with open(OUTPUT,"w",encoding="utf8") as f:

    json.dump(

        [serialize(e) for e in entries],

        f,

        ensure_ascii=False,

        indent=2

    )

print("="*60)
print("PASS 5.1 COMPLETE")
print("="*60)
print("Entries :",len(entries))
print("Output  :",OUTPUT)

PASS 5.1 COMPLETE
Entries : 2616
Output  : output/05_1_detector_objects.json


In [ ]:
# ============================================================
# PASS 5.2
# FINAL WORD -> MEANING EXTRACTION
#
# INPUT
# output/05_1_entries.json
#
# OUTPUT
# output/05_2_dictionary.json
#
# Only returns:
#
# {
#     "word": "...",
#     "meaning":[...]
# }
# ============================================================

import json
import re
from pathlib import Path
from tqdm import tqdm

OUTPUT = Path("output")

INPUT = OUTPUT / "05_1_entries.json"
OUT = OUTPUT / "05_2_dictionary.json"

with open(INPUT,"r",encoding="utf8") as f:
    entries = json.load(f)

# --------------------------------------------------------
# metadata
# --------------------------------------------------------

metadata = re.compile(
    r"\|\|.*$|¶.*$"
)

# remove page references

refs = re.compile(
    r"AA\.\#\d+|B\.\d+(\s*\([^)]+\))?"
)

# grammatical tags

grammar = re.compile(
    r"\b("
    r"ADJ|ADV|CONJ|INTERJ|NUM|PRN|"
    r"INTR|TR|IPFV|PFV|CP|NEG|"
    r"SG|PL|DOUBLE|HM|HF|HS|"
    r"HZ|NG|RF|DIST|PROX|ERG|"
    r"GEN|DAT|ABL|ADE|ONO|"
    r"X|Y|Z|H"
    r")\b"
)

# --------------------------------------------------------
# first english detector
# --------------------------------------------------------

english_start = re.compile(
    r"\b("
    r"a|an|the|to|be|become|go|come|put|"
    r"sit|stand|keep|live|stay|leave|"
    r"burn|melt|drink|eat|see|look|"
    r"move|run|walk|pull|throw|"
    r"house|home|woman|man|child|"
    r"water|fire|stone|tree|animal|"
    r"intelligent|smart|young|small|large|"
    r"thick|thin|red|white|black|"
    r"uphill|downhill|forest|village|"
    r"lake|river|mountain|"
    r"[A-Za-z]{3,}"
    r")\b"
)

# --------------------------------------------------------
# cleaning
# --------------------------------------------------------

def clean_word(text):

    text = refs.sub(" ",text)

    text = metadata.sub("",text)

    text = grammar.sub(" ",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip(" -;,|")



def clean_meaning(text):

    text = metadata.sub("",text)

    text = refs.sub("",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip(" -;,|")

# --------------------------------------------------------
# extraction
# --------------------------------------------------------

dictionary = []

failed = 0

for entry in tqdm(entries):

    text = " ".join(entry["raw_lines"])

    text = re.sub(r"\s+"," ",text)

    m = english_start.search(text)

    if not m:

        failed += 1
        continue

    left = text[:m.start()]

    right = text[m.start():]

    word = clean_word(left)

    meaning = clean_meaning(right)

    if len(word) < 1:
        failed += 1
        continue

    meanings = [
        x.strip()
        for x in re.split(r";|,",meaning)
        if len(x.strip()) > 0
    ]

    dictionary.append({

        "word":word,

        "meaning":meanings

    })

# --------------------------------------------------------
# save
# --------------------------------------------------------

with open(OUT,"w",encoding="utf8") as f:

    json.dump(
        dictionary,
        f,
        ensure_ascii=False,
        indent=4
    )

print("="*60)

print("PASS 5.2 COMPLETE")

print("="*60)

print("Dictionary entries :",len(dictionary))

print("Failed :",failed)

print()

print("Generated")

print(OUT)